# GPT-2 Baseline (Colab)

Fine-tuned `gpt2` sentence classifier with:
- **same split** as `bert_attention_pipeline.ipynb` (loaded from `bert_bias_classifier_v9_split.npz`)
- train_fit / val / test partition (val for early stopping)
- test set restricted to real/original sentences
- multi-seed evaluation (different weight init, same data split)
- optional LOSO
- checkpoint save/load

In [3]:
!pip -q install --upgrade transformers torch torchvision scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 89.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have panda

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
)

from transformers import (
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    get_linear_schedule_with_warmup,
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to your Drive folder
ROOT = Path('/content/drive/MyDrive/attention-atlas-colab')
ROOT.mkdir(parents=True, exist_ok=True)

DATA_JSON = ROOT / 'bias_sentences_v9.json'
FEAT_PKL = ROOT / 'feature_matrix_gpt2_v9.pkl'
SPLIT_NPZ = ROOT / 'bert_bias_classifier_v9_split.npz'

OUT_DIR = ROOT / 'gpt2_baseline_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_JSON:', DATA_JSON)
print('FEAT_PKL :', FEAT_PKL)
print('SPLIT_NPZ:', SPLIT_NPZ)
print('OUT_DIR  :', OUT_DIR)

Mounted at /content/drive
DATA_JSON: /content/drive/MyDrive/attention-atlas-colab/bias_sentences_v9.json
FEAT_PKL : /content/drive/MyDrive/attention-atlas-colab/feature_matrix_gpt2_v9.pkl
SPLIT_NPZ: /content/drive/MyDrive/attention-atlas-colab/bert_bias_classifier_v9_split.npz
OUT_DIR  : /content/drive/MyDrive/attention-atlas-colab/gpt2_baseline_outputs


In [3]:
with open(DATA_JSON, encoding='utf-8') as f:
    raw = json.load(f)

df_sentences = pd.DataFrame(raw['entries']).copy()
df_sentences['label'] = df_sentences['has_bias'].astype(int)

SOURCE_CANONICAL = {
    'biased_corpus_only': 'biased-corpus',
    'biased_corpus_v2': 'biased-corpus',
    'gemini_only': 'gemini',
    'gemini_only_v2': 'gemini',
    'gus_only': 'gus-dataset',
    'gus_only_v2': 'gus-dataset',
}
df_sentences['source_canonical'] = (
    df_sentences['source'].map(SOURCE_CANONICAL).fillna(df_sentences['source'])
)

df_features = pd.read_pickle(FEAT_PKL).copy()

if len(df_features) != len(df_sentences):
    raise RuntimeError(f'Size mismatch: features={len(df_features)} vs json={len(df_sentences)}')

for col in [
    'text', 'source', 'source_canonical', 'original_id', 'role', 'topic',
    'pair_id', 'sentence_id', 'edit_type',
]:
    if col in df_sentences.columns:
        df_features[col] = df_sentences[col].values

y = df_features['label'].astype(int)
texts = df_features['text'].values
sources = df_features['source_canonical'].values
unique_sources = np.array(sorted(np.unique(sources)))

# ── Load persisted split (same as bert_attention_pipeline.ipynb) ──
split = np.load(SPLIT_NPZ)
train_idx = split['train_idx']
test_idx = split['test_idx']
train_fit_idx = split['train_fit_idx']
val_idx = split['val_idx']

print('N:', len(df_features))
print('Label dist:', y.value_counts().to_dict())
print('Sources:', pd.Series(sources).value_counts().to_dict())
print(f'\nLoaded split from {SPLIT_NPZ.name}:')
print(f'  train_fit: {len(train_fit_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')
print(f'  train (fit+val): {len(train_idx)}')

N: 10304
Label dist: {1: 5497, 0: 4807}
Sources: {'gemini': 4440, 'biased-corpus': 3235, 'gus-dataset': 2629}

Loaded split from bert_bias_classifier_v9_split.npz:
  train_fit: 6580  val: 1649  test: 1282
  train (fit+val): 8229


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BASELINE_SEEDS = [1, 2, 3, 4, 5]  # set [1,2,3] if you need lighter runs
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-5

RUN_LOSO = True
USE_SAVED_MODELS = True
SAVE_MODELS = True

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def encode_texts(text_array, tok, max_len):
    enc = tok(
        list(text_array),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']


def make_gpt2_model():
    model = GPT2ForSequenceClassification.from_pretrained('gpt2', num_labels=2)
    model.config.pad_token_id = tokenizer.pad_token_id
    return model


def train_gpt2_classifier(X_ids, X_mask, y_train,
                          X_ids_val, X_mask_val, y_val,
                          seed, epochs=EPOCHS, lr=LR, patience=3):
    """Train with validation-loss early stopping."""
    set_all_seeds(seed)
    model = make_gpt2_model().to(device)

    dataset = TensorDataset(
        X_ids.to(device),
        X_mask.to(device),
        torch.tensor(y_train, dtype=torch.long).to(device),
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    val_dataset = TensorDataset(
        X_ids_val.to(device),
        X_mask_val.to(device),
        torch.tensor(y_val, dtype=torch.long).to(device),
    )
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    best_val_loss = float('inf')
    best_state = None
    bad_epochs = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for ids, mask, labels in loader:
            optimizer.zero_grad()
            outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation loss
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for ids, mask, labels in val_loader:
                outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
                val_loss += outputs.loss.item()
        val_loss /= len(val_loader)

        print(f'    Epoch {epoch+1}/{epochs}  loss={total_loss/len(loader):.4f}  val_loss={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f'    Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    return model


def predict_gpt2(model, X_ids, X_mask):
    model.eval()
    dataset = TensorDataset(X_ids.to(device), X_mask.to(device))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    all_probs = []
    with torch.no_grad():
        for ids, mask in loader:
            outputs = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs)


def ckpt_main(seed):
    return OUT_DIR / f'gpt2_seed_{seed}'


def ckpt_loso(seed, src):
    safe_src = str(src).replace('/', '_')
    return OUT_DIR / f'gpt2_seed_{seed}_loso_{safe_src}'

In [6]:
gpt2_ft_seed_results = []
gpt2_ft_seed_loso = []

# Encode val and test once (same across all seeds)
ids_val, mask_val = encode_texts(texts[val_idx], tokenizer, MAX_LEN)
ids_te, mask_te = encode_texts(texts[test_idx], tokenizer, MAX_LEN)
y_val_arr = y.iloc[val_idx].values
y_te = y.iloc[test_idx].values

# Encode train_fit once
ids_tr, mask_tr = encode_texts(texts[train_fit_idx], tokenizer, MAX_LEN)
y_tr = y.iloc[train_fit_idx].values

for seed in BASELINE_SEEDS:
    print('\n' + '='*60)
    print(f'Fine-tuned GPT-2 - SEED = {seed}')
    print('='*60)

    main_dir = ckpt_main(seed)
    if USE_SAVED_MODELS and main_dir.exists():
        print(f'  Loading checkpoint: {main_dir}')
        model = GPT2ForSequenceClassification.from_pretrained(main_dir).to(device)
    else:
        model = train_gpt2_classifier(
            ids_tr, mask_tr, y_tr,
            ids_val, mask_val, y_val_arr,
            seed,
        )
        if SAVE_MODELS:
            main_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(main_dir)
            tokenizer.save_pretrained(main_dir)
            print(f'  Saved checkpoint: {main_dir}')

    probs_te = predict_gpt2(model, ids_te, mask_te)
    preds_te = (probs_te >= 0.5).astype(int)

    acc_b  = accuracy_score(y_te, preds_te)
    f1_b   = f1_score(y_te, preds_te, zero_division=0)
    prec_b = precision_score(y_te, preds_te, zero_division=0)
    rec_b  = recall_score(y_te, preds_te, zero_division=0)
    auc_b  = roc_auc_score(y_te, probs_te)

    # Val metrics
    probs_val = predict_gpt2(model, ids_val, mask_val)
    preds_val = (probs_val >= 0.5).astype(int)
    f1_val = f1_score(y_val_arr, preds_val, zero_division=0)

    print(f'  Train-fit={len(train_fit_idx)}  Val={len(val_idx)}  Test={len(test_idx)}')
    print(f'  Val  F1={f1_val:.4f}')
    print(f'  Test Acc={acc_b:.4f}  F1={f1_b:.4f}  Prec={prec_b:.4f}  Rec={rec_b:.4f}  AUC={auc_b:.4f}')

    loso_mean_f1 = np.nan
    loso_mean_acc = np.nan
    loso_mean_auc = np.nan
    loso_mean_prec = np.nan
    loso_mean_rec = np.nan
    if RUN_LOSO:
        loso_accs, loso_f1s, loso_aucs, loso_precs, loso_recs = [], [], [], [], []
        for test_src in unique_sources:
            tmask = sources == test_src
            tr_l = np.where(~tmask)[0]
            te_l = np.where(tmask)[0]

            # LOSO val: 20% of train
            from sklearn.model_selection import GroupShuffleSplit
            loso_pairs = df_features.iloc[tr_l]['pair_id'].copy()
            loso_na = loso_pairs.isna()
            loso_pairs[loso_na] = [f'unpaired_{i}' for i in range(loso_na.sum())]
            gss_lv = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
            tr_lv, val_lv = next(gss_lv.split(tr_l, y.iloc[tr_l], groups=loso_pairs.values))
            tr_l_fit = tr_l[tr_lv]
            val_l = tr_l[val_lv]

            ids_tr_l, mask_tr_l = encode_texts(texts[tr_l_fit], tokenizer, MAX_LEN)
            ids_val_l, mask_val_l = encode_texts(texts[val_l], tokenizer, MAX_LEN)
            ids_te_l, mask_te_l = encode_texts(texts[te_l], tokenizer, MAX_LEN)

            loso_dir = ckpt_loso(seed, test_src)
            if USE_SAVED_MODELS and loso_dir.exists():
                print(f'    [LOSO {test_src}] loading: {loso_dir}')
                model_l = GPT2ForSequenceClassification.from_pretrained(loso_dir).to(device)
            else:
                model_l = train_gpt2_classifier(
                    ids_tr_l, mask_tr_l, y.iloc[tr_l_fit].values,
                    ids_val_l, mask_val_l, y.iloc[val_l].values,
                    seed,
                )
                if SAVE_MODELS:
                    loso_dir.mkdir(parents=True, exist_ok=True)
                    model_l.save_pretrained(loso_dir)
                    tokenizer.save_pretrained(loso_dir)
                    print(f'    [LOSO {test_src}] saved: {loso_dir}')

            probs_l = predict_gpt2(model_l, ids_te_l, mask_te_l)
            preds_l = (probs_l >= 0.5).astype(int)
            y_te_l = y.iloc[te_l].values

            acc_l  = accuracy_score(y_te_l, preds_l)
            f1_l   = f1_score(y_te_l, preds_l, zero_division=0)
            prec_l = precision_score(y_te_l, preds_l, zero_division=0)
            rec_l  = recall_score(y_te_l, preds_l, zero_division=0)
            # AUC can be undefined if a LOSO fold has only one class - guard it.
            try:
                auc_l = roc_auc_score(y_te_l, probs_l)
            except ValueError:
                auc_l = np.nan

            loso_accs.append(acc_l)
            loso_f1s.append(f1_l)
            loso_aucs.append(auc_l)
            loso_precs.append(prec_l)
            loso_recs.append(rec_l)

            gpt2_ft_seed_loso.append({
                'seed': seed, 'source': test_src,
                'accuracy': acc_l,
                'f1': f1_l,
                'precision': prec_l,
                'recall': rec_l,
                'auc': auc_l,
            })

            print(f'    [LOSO {test_src}] Acc={acc_l:.4f}  F1={f1_l:.4f}  '
                  f'Prec={prec_l:.4f}  Rec={rec_l:.4f}  AUC={auc_l if not np.isnan(auc_l) else float("nan"):.4f}')

            del model_l
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        loso_mean_f1   = float(np.mean(loso_f1s))
        loso_mean_acc  = float(np.mean(loso_accs))
        loso_mean_auc  = float(np.nanmean(loso_aucs))
        loso_mean_prec = float(np.mean(loso_precs))
        loso_mean_rec  = float(np.mean(loso_recs))
        print(f'  LOSO mean  Acc={loso_mean_acc:.4f}  F1={loso_mean_f1:.4f}  '
              f'Prec={loso_mean_prec:.4f}  Rec={loso_mean_rec:.4f}  AUC={loso_mean_auc:.4f}')

    gpt2_ft_seed_results.append({
        'seed': seed,
        'accuracy': acc_b,
        'f1': f1_b,
        'precision': prec_b,
        'recall': rec_b,
        'auc': auc_b,
        'val_f1': f1_val,
        'loso_mean_accuracy': loso_mean_acc,
        'loso_mean_f1': loso_mean_f1,
        'loso_mean_precision': loso_mean_prec,
        'loso_mean_recall': loso_mean_rec,
        'loso_mean_auc': loso_mean_auc,
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_gpt2_ft = pd.DataFrame(gpt2_ft_seed_results)
df_gpt2_loso = pd.DataFrame(gpt2_ft_seed_loso)

display(df_gpt2_ft)
if not df_gpt2_loso.empty:
    display(df_gpt2_loso.head())



Fine-tuned GPT-2 - SEED = 1


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

    Epoch 1/20  loss=1.5434  val_loss=0.2270
    Epoch 2/20  loss=0.2212  val_loss=0.1099
    Epoch 3/20  loss=0.1327  val_loss=0.0946
    Epoch 4/20  loss=0.0871  val_loss=0.0824
    Epoch 5/20  loss=0.0524  val_loss=0.1130
    Epoch 6/20  loss=0.0408  val_loss=0.1438
    Epoch 7/20  loss=0.0198  val_loss=0.1895
    Early stopping at epoch 7 (best val_loss=0.0824)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    [LOSO gus-dataset] saved: /content/drive/MyDrive/attention-atlas-colab/gpt2_baseline_outputs/gpt2_seed_5_loso_gus-dataset
  LOSO mean F1=0.9089


,seed,accuracy,f1,auc,val_f1,loso_mean_f1
0,1,0.953198,0.944751,0.991004,0.967927,0.901220
1,2,0.955538,0.948229,0.992754,0.955342,0.906680
2,3,0.954758,0.948307,0.992806,0.963949,0.912550
3,4,0.951638,0.944742,0.993524,0.964206,0.898704
4,5,0.958658,0.951949,0.991484,0.957907,0.908889


,seed,source,f1
0,1,biased-corpus,0.939424
1,1,gemini,0.854049
2,1,gus-dataset,0.910189
3,2,biased-corpus,0.934473
4,2,gemini,0.865506


In [7]:
print('\n' + '='*60)
print(f'Fine-tuned GPT-2 SUMMARY - {len(BASELINE_SEEDS)} seeds')
print('='*60)

_summary_cols = [
    'accuracy', 'f1', 'precision', 'recall', 'auc',
    'val_f1',
    'loso_mean_accuracy', 'loso_mean_f1',
    'loso_mean_precision', 'loso_mean_recall', 'loso_mean_auc',
]
for col in _summary_cols:
    if col in df_gpt2_ft.columns and df_gpt2_ft[col].notna().any():
        m, s = df_gpt2_ft[col].mean(), df_gpt2_ft[col].std()
        print(f'  {col:22s}: {m:.4f} +/- {s:.4f}')

# Per-source LOSO summary (mean +/- std across seeds)
if not df_gpt2_loso.empty:
    print('\nLOSO per source (mean +/- std across seeds):')
    _per_src = (
        df_gpt2_loso
        .groupby('source')[['accuracy', 'f1', 'precision', 'recall', 'auc']]
        .agg(['mean', 'std'])
        .round(4)
    )
    print(_per_src.to_string())

df_gpt2_ft.to_csv(OUT_DIR / 'df_gpt2_ft.csv', index=False)
if not df_gpt2_loso.empty:
    df_gpt2_loso.to_csv(OUT_DIR / 'df_gpt2_loso.csv', index=False)

print('\nSaved:')
print(' -', OUT_DIR / 'df_gpt2_ft.csv')
if not df_gpt2_loso.empty:
    print(' -', OUT_DIR / 'df_gpt2_loso.csv')



Fine-tuned GPT-2 SUMMARY - 5 seeds
  accuracy       : 0.9548 +/- 0.0026
  f1             : 0.9476 +/- 0.0030
  auc            : 0.9923 +/- 0.0010
  val_f1         : 0.9619 +/- 0.0051
  loso_mean_f1   : 0.9056 +/- 0.0056

Saved:
 - /content/drive/MyDrive/attention-atlas-colab/gpt2_baseline_outputs/df_gpt2_ft.csv
 - /content/drive/MyDrive/attention-atlas-colab/gpt2_baseline_outputs/df_gpt2_loso.csv


In [ ]:
# ── Save best fine-tuned GPT-2 ──
import shutil

best_row = df_gpt2_ft.loc[df_gpt2_ft['f1'].idxmax()]
best_seed = int(best_row['seed'])
best_dir = ckpt_main(best_seed)

FINAL_DIR = OUT_DIR / 'gpt2_best'
if best_dir.exists():
    if FINAL_DIR.exists():
        shutil.rmtree(FINAL_DIR)
    shutil.copytree(best_dir, FINAL_DIR)
    print(f'Best seed: {best_seed} (F1={best_row["f1"]:.4f})')
    print(f'Saved to: {FINAL_DIR}')
else:
    print(f'Checkpoint not found: {best_dir}')